# 실습 문제 - 340. Skills Pattern

`# TODO` 표시된 부분을 채우며 위에서부터 셀 단위로 실행하세요. (모범답안: `practice_answer_340_Skills_Pattern.ipynb`)

### 문제 요약

본문의 Skills Pattern(Progressive Disclosure)을 새로운 도메인에 그대로 적용합니다.

1. 사내 규정 스킬 2개 정의: `vacation_policy`, `expense_policy`
2. 스킬 content를 반환하는 `load_company_skill` 도구
3. 스킬 설명만 시스템 프롬프트에 추가하는 미들웨어 + 에이전트 생성 후 테스트

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
import os
from langchain.chat_models import init_chat_model

model = init_chat_model(
    os.environ["OPENAI_MODEL"],
    model_provider="openai",
    use_responses_api=True,        # Responses API 사용 (Chat Completions에서는 추론 모델의 도구 호출 불가)
    reasoning={"effort": "low"},   # 추론 강도 낮춤 → 실습 속도·비용 절감
)

## 1. 사내 규정 스킬 2개 정의

본문의 `Skill` 구조(name, description, content)를 그대로 사용합니다.
description은 시스템 프롬프트에 노출되고, content는 도구 호출 시에만 로드됩니다.

In [ ]:
from typing import TypedDict

class Skill(TypedDict):
    """에이전트가 필요할 때 로딩할 수 있는 스킬 정의 구조"""
    name: str
    description: str
    content: str

MY_SKILLS: list[Skill] = [
    {
        "name": "vacation_policy",
        "description": "휴가 규정 (연차, 반차, 신청 절차).",
        "content": """# 휴가 규정
- 연차: 1년 근속 시 15일 부여, 최대 25일까지 누적
- 반차: 오전/오후 반차 사용 가능 (0.5일 차감)
- 신청: 사용 3일 전까지 인사 시스템에서 신청, 팀장 승인 필요
""",
    },
    # TODO: 위 vacation_policy 와 같은 형식으로 경비 처리 규정 스킬을 추가하세요.
    #   name: "expense_policy" / description: "경비 처리 규정 (식대, 교통비, 영수증)."
    #   content: 식대 1인 1식 15,000원(야근 시 20,000원), 교통비 실비 정산(택시는 사전 승인),
    #            영수증은 사용일로부터 7일 이내 제출
    {
        "name": ...,
        "description": ...,
        "content": ...,
    },
]

## 2. load_company_skill 도구

본문의 `load_skill`과 같은 구조입니다.
스킬 이름으로 검색해서 content를 반환하고, 없으면 사용 가능한 스킬 목록을 안내합니다.

In [ ]:
from langchain.tools import tool

@tool
def load_company_skill(skill_name: str) -> str:
    """사내 규정 스킬의 전체 내용을 로드합니다."""
    for skill in MY_SKILLS:
        # TODO: 스킬 이름이 요청한 skill_name 과 같은지 비교하세요.
        if ...:
            return f"Loaded skill: {skill_name}\n\n{skill['content']}"

    available = ", ".join(s["name"] for s in MY_SKILLS)
    return f"스킬 '{skill_name}'을 찾을 수 없습니다. 사용 가능: {available}"


## 3. 미들웨어 + 에이전트 생성 및 테스트

본문의 `SkillMiddleware`와 같은 구조로, 시스템 프롬프트에 **스킬 설명만** 추가합니다.
질문하면 에이전트가 `load_company_skill(vacation_policy)`을 호출한 뒤 규정 내용으로 답변합니다.

In [ ]:
from typing import Callable
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse

class CompanySkillMiddleware(AgentMiddleware):
    """시스템 프롬프트에 사내 규정 스킬 설명 목록을 주입하는 미들웨어"""
    # TODO: 이 미들웨어가 에이전트에 추가할 도구 목록을 지정하세요.
    tools = ...

    def __init__(self):
        skills_list = [f"- **{s['name']}**: {s['description']}" for s in MY_SKILLS]
        self.skills_prompt = "\n".join(skills_list)

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        skills_addendum = (
            f"\n\n## 사용 가능한 스킬\n\n{self.skills_prompt}\n\n"
            "질문에 답하기 전에 load_company_skill 도구로 관련 규정의 상세 내용을 로드하세요."
        )
        current = getattr(request, "system_prompt", None) or ""
        # TODO: request.override(system_prompt=...)로 기존 프롬프트 뒤에 skills_addendum 을 붙여 handler 를 호출하세요.
        return ...


agent = create_agent(
    model,
    system_prompt="당신은 사내 규정 안내 도우미입니다. 규정에 근거해서 답변하세요.",
    # TODO: 위에서 만든 미들웨어를 등록하세요.
    middleware=...,
)

In [ ]:
from langchain_core.messages import HumanMessage

# stream_mode="values" 로 새로 생성되는 메시지만 출력
for step in agent.stream(
    {"messages": [HumanMessage(content="연차 휴가는 며칠인가요?")]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

## 확인 포인트

- 시스템 프롬프트에는 두 스킬의 **설명(한 줄)** 만 들어가 있습니다.
- 에이전트가 질문을 보고 `load_company_skill("vacation_policy")`을 호출하면,
  그때서야 휴가 규정 상세 내용이 컨텍스트에 로드됩니다 — 이것이 **Progressive Disclosure**입니다.
- 경비 관련 질문(예: "야근 식대 한도는?")을 던지면 `expense_policy` 스킬이 로드되는 것도 확인해 보세요.